# <font color="#418FDE" size="6.5" uppercase>**Probability Calibration**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Assess probability reliability using calibration curves and proper scoring metrics. 
- Apply sigmoid and isotonic calibration to improve probability estimates. 
- Design calibration workflows that avoid leakage and support multiclass settings. 


## **1. Calibration Metrics**

### **1.1. Reliability Assessment**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_01_01.jpg?v=1783848157" width="250">



>* Reliable probabilities match real-world outcome frequencies.
>* Decisions need trustworthy probabilities, not just rankings.

>* Calibration curves compare predicted and actual rates.
>* Overconfidence and underconfidence change decision usefulness.

>* Reliability depends on sufficient, well-grouped data.
>* Check calibration across groups, times, and conditions.



### **1.2. Proper Scoring Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_01_02.jpg?v=1783848158" width="250">



>* Scores evaluate probability quality, not just correctness.
>* Proper metrics reward honest, evidence-based confidence.

>* Log loss punishes confident wrong predictions heavily.
>* Brier score measures average probability error.

>* Scores reflect calibration and discrimination together.
>* Interpret scores with curves and decisions.



In [ ]:
#@title Python Code - Proper Scoring Metrics

# Proper scores compare probability quality fairly.
# This example contrasts cautious and overconfident forecasts.
# Lower scores mean better probability estimates.

import numpy as np
import matplotlib.pyplot as plt

# Use a deterministic random generator.
rng = np.random.default_rng(7)

# Create binary outcomes with known chances.
true_probs = np.linspace(0.1, 0.9, 120)
y_true = rng.binomial(1, true_probs)

# Build three different probability forecasters.
p_good = true_probs.copy()
p_flat = np.full_like(true_probs, y_true.mean(), dtype=float)
p_over = np.clip(1.6 * true_probs - 0.3, 0.001, 0.999)

# Define a simple Brier score.
def brier_score(y, p):
    return np.mean((p - y) ** 2)

# Define a simple log loss.
def log_loss_score(y, p):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

# Compute scores for each forecaster.
results = {
    "Well calibrated": (brier_score(y_true, p_good), log_loss_score(y_true, p_good)),
    "Always average": (brier_score(y_true, p_flat), log_loss_score(y_true, p_flat)),
    "Overconfident": (brier_score(y_true, p_over), log_loss_score(y_true, p_over)),
}

# Print a compact score summary.
print("Proper scoring metrics on tiny forecasts")
for name, scores in results.items():
    print(f"{name}: Brier={scores[0]:.3f}, LogLoss={scores[1]:.3f}")

# Prepare calibration style bins.
bins = np.linspace(0.0, 1.0, 6)
centers = (bins[:-1] + bins[1:]) / 2

# Compute observed rates by bin.
def observed_rate(y, p, edges):
    rates = []
    for left, right in zip(edges[:-1], edges[1:]):
        mask = (p >= left) & (p < right)
        rates.append(y[mask].mean() if mask.any() else np.nan)
    return np.array(rates)

# Plot one simple reliability comparison.
obs_good = observed_rate(y_true, p_good, bins)
obs_over = observed_rate(y_true, p_over, bins)
plt.figure(figsize=(6, 4))
plt.plot([0, 1], [0, 1], "k--", label="Perfect calibration")

# Add two forecast curves.
plt.plot(centers, obs_good, "o-", label="Well calibrated")
plt.plot(centers, obs_over, "s-", label="Overconfident")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")

# Finish the teaching plot.
plt.title("Calibration and proper scoring metrics")
plt.legend()
plt.tight_layout()
plt.show()



### **1.3. Proper Scoring Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_01_03.jpg?v=1783848160" width="250">



>* Rewards honest, accurate probability estimates.
>* Checks whether confidence levels are trustworthy.

>* Log loss heavily punishes confident mistakes.
>* Brier score smoothly measures calibration quality.

>* Scores reflect calibration and discrimination together.
>* Use curves too for trustworthy decisions.



In [ ]:
#@title Python Code - Proper Scoring Metrics

# Proper scores compare probabilities with outcomes.
# This example contrasts cautious and overconfident forecasts.
# One plot summarizes calibration and score quality.

import numpy as np
import matplotlib.pyplot as plt

# Fix randomness for reproducible teaching results.
rng = np.random.default_rng(7)
true_probs = np.linspace(0.05, 0.95, 200)
y_true = rng.binomial(1, true_probs)

# Build two probability prediction styles.
p_good = np.clip(true_probs + rng.normal(0, 0.04, 200), 0.001, 0.999)
p_bad = np.clip(0.5 + 1.8 * (true_probs - 0.5), 0.001, 0.999)

# Check matching lengths before scoring.
assert y_true.shape == p_good.shape == p_bad.shape
bins = np.linspace(0.0, 1.0, 6)
centers = (bins[:-1] + bins[1:]) / 2

# Define beginner friendly scoring functions.
def log_loss_score(y, p):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

def brier_score(y, p):
    return np.mean((p - y) ** 2)

# Compute simple calibration curve points.
def calibration_points(y, p, edges):
    xs, ys = [], []
    for left, right in zip(edges[:-1], edges[1:]):

        mask = (p >= left) & (p < right)
        if np.any(mask):
            xs.append(p[mask].mean())
            ys.append(y[mask].mean())

    return np.array(xs), np.array(ys)

good_x, good_y = calibration_points(y_true, p_good, bins)
bad_x, bad_y = calibration_points(y_true, p_bad, bins)

# Print compact score comparisons.
print(f"Good model log loss: {log_loss_score(y_true, p_good):.3f}")
print(f"Bad model log loss:  {log_loss_score(y_true, p_bad):.3f}")
print(f"Good model Brier:    {brier_score(y_true, p_good):.3f}")
print(f"Bad model Brier:     {brier_score(y_true, p_bad):.3f}")

# Plot reliability against perfect calibration.
plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
plt.plot(good_x, good_y, "o-", label="Better calibrated")
plt.plot(bad_x, bad_y, "s-", label="Overconfident")

# Add labels and readable limits.
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed event frequency")
plt.title("Calibration curves with proper scoring metrics")
plt.legend()

# Show the single teaching plot.
plt.tight_layout()
plt.show()



## **2. Calibration Methods**

### **2.1. Sigmoid Calibration**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_02_01.jpg?v=1783848155" width="250">



>* Smoothly fixes overconfident or cautious probabilities.
>* Learns corrected probabilities from separate calibration data.

>* Stable calibration reduces overfitting with limited data.
>* Preserves ranking while improving probability realism.

>* Best for smooth, monotonic miscalibration patterns.
>* Use separate data; simple, stable first choice.



### **2.2. Isotonic Calibration**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_02_02.jpg?v=1783848150" width="250">



>* Flexible calibration preserves score ordering.
>* Adjusts distorted probabilities to match outcomes.

>* Monotonic mapping adjusts probabilities by score region.
>* Improves observed probability alignment for decisions.

>* Flexible but risky with small calibration data.
>* Best with ample data and careful validation.



### **2.3. Choosing Calibration Methods**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_02_03.jpg?v=1783848151" width="250">



>* Sigmoid suits small data, smoother errors.
>* Isotonic fits complex errors, needs more data.

>* Use sigmoid for smooth, stable miscalibration.
>* Use isotonic for uneven errors, enough data.

>* Validate both methods on held-out data.
>* Balance stability, data, and decision risks.



## **3. Calibration Workflow Design**

### **3.1. Leakage Safe Calibration**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_03_01.jpg?v=1783848169" width="250">



>* Separate model training from calibration data.
>* Avoid leakage for trustworthy deployed probabilities.

>* Separate training, calibration, and testing stages.
>* Use unseen predictions to avoid overconfidence.

>* Fit preprocessing only within training splits.
>* Respect time, groups, and deployment boundaries.



### **3.2. Leakage Safe Calibration**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_03_02.jpg?v=1783848162" width="250">



>* Separate model training from calibration data.
>* Shared data creates overconfident, unreliable probabilities.

>* Use separate data for training and calibration.
>* Prevent leakage across preprocessing and pipeline steps.

>* Use out-of-fold predictions for calibration.
>* Keep a final test set untouched.



### **3.3. Multiclass Calibration Workflow**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_15/Lecture_B/image_03_03.jpg?v=1783848165" width="250">



>* Multiclass calibration needs reliable class probabilities.
>* Separate training, calibration, and testing data.

>* Calibrate classes separately or jointly, then normalize.
>* Match method to data, especially rare classes.

>* Use cross-validation to prevent calibration leakage.
>* Test calibrated probabilities under realistic conditions.



# <font color="#418FDE" size="6.5" uppercase>**Probability Calibration**</font>


In this lecture, you learned to:
- Assess probability reliability using calibration curves and proper scoring metrics. 
- Apply sigmoid and isotonic calibration to improve probability estimates. 
- Design calibration workflows that avoid leakage and support multiclass settings. 

In the next Module (Module 16), we will go over 'None'